# CFST XGBoost Colab 训练 Notebook

这个 notebook 面向当前仓库的最新训练逻辑：
- `config.model.params` 是唯一超参数来源（严格校验）
- `use_optuna=True` 时会先调优，再在同一轮用最优参数重训
- `use_optuna=False` 时仅在 `context_hash` 匹配时加载 `logs/best_params.json`
- Optuna study 默认持久化到 `logs/optuna_study.db`，可通过 Drive 续跑


In [ ]:
# ===== 用户参数区（先改这里） =====
REPO_URL = "https://github.com/xyf1qgnn-cpu/xgboost.git"
REPO_BRANCH = "main"
WORKDIR = "/content/xgboost"

USE_DRIVE = True
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/cfst_xgboost_colab"

USE_GPU = True
USE_OPTUNA = True
N_TRIALS = 400
OPTUNA_TIMEOUT = 4 * 60 * 60

SOURCE_DATA_PATH = "data/raw/feature_parameters_unique.csv"
BEST_PARAMS_PATH = "logs/best_params.json"
OPTUNA_DB_PATH = "logs/optuna_study.db"

OUTPUT_DIR = "output_colab"
COLAB_CONFIG_PATH = "config/config_colab.yaml"


In [ ]:
# 挂载 Google Drive（用于持久化 logs/output）
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print(f"Drive mounted. Root dir: {DRIVE_PROJECT_DIR}")
else:
    print("USE_DRIVE=False，将只保存在 Colab 临时磁盘")


In [ ]:
# 拉取代码
import os
import shutil

if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)

!git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {WORKDIR}
%cd {WORKDIR}


In [ ]:
# 安装依赖
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt
!python -m pip install -q --upgrade xgboost optuna


In [ ]:
# 从 Drive 恢复上一次调优状态（可选，但推荐）
from pathlib import Path
import shutil

if USE_DRIVE:
    project_dir = Path(DRIVE_PROJECT_DIR)
    project_dir.mkdir(parents=True, exist_ok=True)

    for rel in [BEST_PARAMS_PATH, OPTUNA_DB_PATH]:
        src = project_dir / rel
        dst = Path(rel)
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            print(f"[RESTORE] {src} -> {dst}")
        else:
            print(f"[SKIP] not found in Drive: {src}")
else:
    print("USE_DRIVE=False，跳过恢复历史日志")


In [ ]:
# 生成 Colab 专用配置，避免直接改动 config/config.yaml
from pathlib import Path
import yaml

import torch

with open("config/config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

device = "cuda" if USE_GPU and torch.cuda.is_available() else "cpu"

cfg["data"]["file_path"] = SOURCE_DATA_PATH
cfg["model"]["use_optuna"] = bool(USE_OPTUNA)
cfg["model"]["n_trials"] = int(N_TRIALS)
cfg["model"]["optuna_timeout"] = int(OPTUNA_TIMEOUT)
cfg["model"]["best_params_path"] = BEST_PARAMS_PATH
cfg["model"]["optuna_storage_url"] = f"sqlite:///{OPTUNA_DB_PATH}"
cfg["model"]["params"]["device"] = device
cfg["model"]["params"]["n_jobs"] = 1 if device == "cuda" else -1
cfg["model"]["params"]["tree_method"] = "hist"

colab_cfg_path = Path(COLAB_CONFIG_PATH)
colab_cfg_path.parent.mkdir(parents=True, exist_ok=True)
with open(colab_cfg_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print(f"Device selected: {device}")
print(f"Config written to: {colab_cfg_path}")
print(f"use_optuna={cfg['model']['use_optuna']}, n_trials={cfg['model']['n_trials']}")
print(f"data.file_path={cfg['data']['file_path']}")


In [ ]:
# 环境与数据检查
import pandas as pd
import torch

print(f"Torch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

df = pd.read_csv(SOURCE_DATA_PATH)
print(f"Dataset shape: {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
display(df.head(3))


In [ ]:
# 开始训练（实时日志 + ETA）
import os
import re
import subprocess
import time

cmd = [
    "python",
    "-u",
    "train.py",
    "--config",
    COLAB_CONFIG_PATH,
    "--output",
    OUTPUT_DIR,
]
print("Run command:", " ".join(cmd))

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"

trial_re = re.compile(r"Trial\s+(\d+)\s+finished")
optuna_start = None
first_trial_no = None
last_trial_no = None

start = time.time()
process = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

if process.stdout is None:
    raise RuntimeError("Failed to capture training stdout")

for raw_line in process.stdout:
    line = raw_line.rstrip("\n")
    print(line, flush=True)

    m = trial_re.search(line)
    if m:
        trial_no = int(m.group(1))
        if first_trial_no is None:
            first_trial_no = trial_no
            optuna_start = time.time()
        last_trial_no = trial_no

        completed = last_trial_no - first_trial_no + 1
        if completed > 0 and optuna_start is not None and N_TRIALS > 0:
            elapsed_opt = time.time() - optuna_start
            sec_per_trial = elapsed_opt / completed
            remaining = max(0, N_TRIALS - completed)
            eta_sec = sec_per_trial * remaining
            print(
                f"[Optuna ETA] completed={completed}/{N_TRIALS}, avg={sec_per_trial:.1f}s/trial, eta~{eta_sec/60:.1f} min",
                flush=True,
            )

return_code = process.wait()
elapsed = time.time() - start
print(f"Training finished with code={return_code}, elapsed={elapsed/60:.1f} min ({elapsed:.1f}s)")

if return_code != 0:
    raise RuntimeError("Training failed, please inspect logs above")


In [ ]:
# 读取核心结果
import json
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
report_path = output_dir / "evaluation_report.json"
meta_path = output_dir / "training_metadata.json"

report = json.loads(report_path.read_text(encoding="utf-8"))
metadata = json.loads(meta_path.read_text(encoding="utf-8"))

print("=== Training Summary ===")
print("params_source:", metadata.get("params_source"))
print("context_hash:", metadata.get("context_hash"))
print("optuna_run_info:", metadata.get("optuna_run_info"))

test_metrics = report.get("test_metrics_original_space", {})
print("=== Test Metrics (original space) ===")
print(f"RMSE: {test_metrics.get('rmse')}")
print(f"MAE : {test_metrics.get('mae')}")
print(f"R2  : {test_metrics.get('r2')}")
print(f"COV : {test_metrics.get('cov')}")


In [ ]:
# 保存本轮产物到 Drive，并同步 logs 以支持下次续跑
from pathlib import Path
import datetime
import shutil

if USE_DRIVE:
    project_dir = Path(DRIVE_PROJECT_DIR)
    run_tag = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = project_dir / "runs" / run_tag
    run_dir.mkdir(parents=True, exist_ok=True)

    # 保存本轮结果快照
    shutil.copytree(OUTPUT_DIR, run_dir / OUTPUT_DIR, dirs_exist_ok=True)
    if Path("logs").exists():
        shutil.copytree("logs", run_dir / "logs", dirs_exist_ok=True)
    shutil.copy2(COLAB_CONFIG_PATH, run_dir / Path(COLAB_CONFIG_PATH).name)

    # 同步关键状态到固定路径（用于下轮恢复）
    for rel in [BEST_PARAMS_PATH, OPTUNA_DB_PATH]:
        src = Path(rel)
        if src.exists():
            dst = project_dir / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            print(f"[SYNC] {src} -> {dst}")

    print(f"Saved run snapshot to: {run_dir}")
else:
    print("USE_DRIVE=False，跳过持久化")


## 续跑建议

- 同一数据上下文继续调优：保持 `SOURCE_DATA_PATH` 和特征配置不变，`USE_OPTUNA=True`，并确保已从 Drive 恢复 `logs/optuna_study.db`。
- 仅用历史最优参数快速训练：设 `USE_OPTUNA=False`，训练脚本会在 `context_hash` 匹配时自动加载 `logs/best_params.json`。
- 如果你替换了数据文件或关键配置，`context_hash` 会变化，旧参数会被安全忽略。
